# Spark X2.6 4B Recursive72 — Loop72 Differentiable `TwoStreamMHC` Training & Hugging Face Upload (Colab G4 96GB)

**Target Model Name:** `Spark X2.6 4B Recursive72` (`TassP0lar/Spark-X2.6-4B-Recursive72` / `<your_hf_user>/Spark-X2.6-4B-Recursive72`)  
**Branch:** `loop72-dlsa` | **Backbone:** `XHToken/Spark-X2.6-4B` (Frozen BF16, 36 physical layers $\rightarrow$ 72 virtual visits: `4 + 24 + 24 + 12 + 8`)

This notebook executes the full **Milestone 1 + Milestone 2** workflow from [`docs/roadmap/09_LOOP72_MHC_TRAINING_AND_VALIDATION.md`](docs/roadmap/09_LOOP72_MHC_TRAINING_AND_VALIDATION.md) and uploads **`Spark X2.6 4B Recursive72`** to Hugging Face Hub upon completion:
1. **Environment & Preflight Verification**: Auto-detects **Colab G4 96 GB VRAM** (`colab_g4_96gb`), checks `HF_TOKEN` (from Colab Secrets or environment), and runs unit tests.
2. **Stage A (Mechanical & Gradient Verification)**: Verifies bounded-gain `TwoStreamMHC` Schema v2 (`gain_mode="sigmoid"`, `mhc_schema_version=2`, $g \in [0, 1]$, $(1-g)I + gP \in \mathcal{B}_2$), `10,078,176` trainable controller parameters, non-zero P2/P3/adapter gradients, frozen base SHA-256 invariance, and checkpoint save/load parity.
3. **Stages B, C, D, E & Automatic Hugging Face Upload (`train_spark_mhc.py`)**:
   - **Stage B**: 36L $\rightarrow$ 72L teacher retention warmup with memory-safe sampled-position full-vocabulary KL ($K=64$, $\beta=0.5$, $T=1.5$).
   - **Stage C**: Multi-pass deliberation training (`fixed_72` default, staged controller unfreezing, `adapter_lr=2e-4`, `routing_lr=3e-5`, 5-domain mixture), saving `stage_c_dense_mhc/loop_v03_state.pt`.
   - **Stage D**: Optional `Oh My Pi` (`omp`) native `<tool_call>` (`read`/`edit`/`bash`) trajectory polish, saving `stage_d_omp_mhc/loop_v03_state.pt`.
   - **Stage E**: 18-visit `VisitAwareDLSAIndexBank` PCA + sampled-query KL distillation on the trained `learned_mhc` residual stream with `mhc_checkpoint_sha256` provenance lock.
   - **Hugging Face Hub Export**: Automatically packages and uploads **`Spark X2.6 4B Recursive72`** (`README.md`, `loop_v03_state.pt`, `visit_selectors_stage_e.pt`, `loop_config_v03.json`, standalone 36L backbone + tokenizer, and runtime scripts) to Hugging Face Hub.
4. **Post-Training Paged D-LSA Evaluation & Decode Speed Benchmark**: Loads the exported `Spark X2.6 4B Recursive72` package into the paged inference engine and runs steady-state decode speed and pinned benchmark suites.

In [ ]:
# === Cell 1: Clone/Update Repository, Check Colab G4 96GB GPU & Verify HF_TOKEN ===
import os, subprocess, sys

REPO_URL = "https://github.com/BaiTian6641/Spark-X2.6-Recursion.git"
BRANCH = "loop72-dlsa"
REPO_DIR = "/content/Spark-X2.6-Recursion" if os.path.exists("/content") else os.getcwd()

if os.path.exists("/content"):
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-r", "requirements-omp.txt"], check=True)

import torch
from train_spark_mhc import detect_gpu_training_profile, normalize_hf_repo_id, resolve_hf_token

hw_profile = detect_gpu_training_profile()
hf_token = resolve_hf_token()
target_repo_id = normalize_hf_repo_id("Spark X2.6 4B Recursive72", token=hf_token)
print("Detected Hardware Profile:", hw_profile)
print("HF_TOKEN detected:", bool(hf_token), "| Target HF Repo:", target_repo_id)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], check=True)

In [ ]:
# === Cell 2: Load Differentiable 72-Visit Training Model & Run Stage A Verification ===
import json
import torch
from load_inference_v03 import load_sparkx_loop72_lsa
from train_spark_mhc import (
    DEFAULT_HF_MODEL_NAME,
    DEFAULT_MODEL_ID,
    DEFAULT_MODEL_REVISION,
    MHCTrainingConfig,
    apply_hardware_profile_to_config,
    normalize_hf_repo_id,
    resolve_hf_token,
    run_stage_a_verification,
)

# Set HF_REPO_ID to 'Spark X2.6 4B Recursive72' (auto-resolves to '<your_hf_username>/Spark-X2.6-4B-Recursive72'
# or defaults to 'TassP0lar/Spark-X2.6-4B-Recursive72'), or specify 'your_org/Spark-X2.6-4B-Recursive72'.
HF_TOKEN = resolve_hf_token()
HF_REPO_ID = normalize_hf_repo_id(os.environ.get("HF_REPO_ID", "Spark X2.6 4B Recursive72"), token=HF_TOKEN)

cfg = MHCTrainingConfig(
    model_id=DEFAULT_MODEL_ID,
    revision=DEFAULT_MODEL_REVISION,
    output_dir="results/mhc_colab_g4_run",
    hardware_profile="auto",  # Automatically selects 'colab_g4_96gb' on >=85GB VRAM
    gain_mode="sigmoid",
    init_p2_gain=0.01,
    init_p3_gain=0.005,
    extra_pass_gate_multiplier=1.0,
    curriculum="fixed_72",
    kl_beta=0.5,
    kl_temperature=1.5,
    sampled_kl_positions=64,
    stage_b_steps=40,
    stage_c_steps=120,
    stage_d_steps=30,
    stage_e_distill_steps=120,
    run_stage_d_omp=True,
    run_stage_e_dlsa=True,
    hf_model_name=DEFAULT_HF_MODEL_NAME,  # 'Spark X2.6 4B Recursive72'
    hf_repo_id=HF_REPO_ID,
    push_to_hub=bool(HF_TOKEN),           # Automatically uploads when HF_TOKEN is present
    hf_token=HF_TOKEN,
    export_standalone_backbone=True,
)
hw_info = apply_hardware_profile_to_config(cfg)
print("Active MHCTrainingConfig Hardware Sizing:", json.dumps(hw_info, indent=2))
print(f"Target Hugging Face Model: '{cfg.hf_model_name}' -> {cfg.hf_repo_id} (push_to_hub={cfg.push_to_hub})")

train_model, train_tokenizer = load_sparkx_loop72_lsa(
    repo_or_dir=cfg.model_id,
    base_repo=cfg.model_id,
    revision=cfg.revision,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    recurrence_mode="learned_mhc",
    attach_sparse_attention=False,  # Mandatory for differentiable training
)

stage_a_report = run_stage_a_verification(
    train_model,
    cfg=cfg,
    vocab_size=int(getattr(train_model.base.config, "vocab_size", 131072)),
    seq_len=64,
)
print("=== Stage A Mechanical & Gradient Verification Report ===")
print(json.dumps(stage_a_report, indent=2, default=str))
assert stage_a_report["passed"], "Stage A verification failed!"

In [ ]:
# === Cell 3: Execute Stages B, C, D, E & Upload 'Spark X2.6 4B Recursive72' to Hugging Face Hub ===
from train_spark_mhc import run_mhc_training_pipeline

# For a quick interactive pilot on Colab G4, adjust step counts below or keep the full defaults from Cell 2:
cfg.seq_len = min(cfg.seq_len, 4096)  # Set to 8192 or 16384 on Colab G4 96GB for long-horizon training

pipeline_report = run_mhc_training_pipeline(
    train_model,
    train_tokenizer,
    cfg=cfg,
    run_stage_a=False,  # Already verified in Cell 2
    run_stage_b=True,
    run_stage_c=True,
    run_stage_d=True,
    run_stage_e=True,
)

print("=== Completed Stages B-E Summary ===")
for stage_key, stage_val in pipeline_report["stages"].items():
    if isinstance(stage_val, dict):
        summary_view = {k: v for k, v in stage_val.items() if k not in ("per_visit_metrics", "checkpoint", "final_export")}
        print(f"\n[{stage_key}]", json.dumps(summary_view, indent=2, default=str))

print("\n=== Hugging Face Hub Export ('Spark X2.6 4B Recursive72') ===")
print(json.dumps(pipeline_report.get("huggingface_export", {}), indent=2, default=str))

In [ ]:
# === Cell 4: Reload Trained `Spark X2.6 4B Recursive72` for Paged Inference & Speed Benchmark ===
import gc
from pathlib import Path
from sparkx_loop72_dlsa_benchmarks import benchmark_loop72_decode_speed

# Free training graph before loading the paged inference instance
del train_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

final_export_dir = str(Path(cfg.output_dir) / "final_loop72_mhc_dlsa_export")
selector_ckpt = str(Path(cfg.output_dir) / "visit_selectors_stage_e.pt")

eval_model, eval_tokenizer = load_sparkx_loop72_lsa(
    repo_or_dir=final_export_dir,
    base_repo=cfg.model_id,
    revision=cfg.revision,
    recurrence_mode="learned_mhc",
    dlsa_checkpoint=selector_ckpt,
    attach_sparse_attention=True,
)
eval_model.eval()

manifest = eval_model.execution_manifest()
print("=== Paged Inference Execution Manifest (Spark X2.6 4B Recursive72) ===")
print(json.dumps(manifest, indent=2, default=str))
assert manifest["effective_recurrence_mode"] == "learned_mhc"
assert manifest["mhc_gain_mode"] == "sigmoid"
assert manifest["mhc_schema_version"] == 2
assert manifest["dlsa_selector"]["recurrence_mode_match"] is True

speed_report = benchmark_loop72_decode_speed(
    eval_model,
    eval_tokenizer,
    prompt_lengths=(512, 2048, 8192),
    gen_tokens=128,
    warmups=1,
    repeats=2,
)
print("=== Steady-State Decode Speed Report ===")
print(json.dumps(speed_report["by_horizon"], indent=2, default=str))